# ASIC Pipeline: Final Harmonized Data

This notebook runs the current ASIC harmonization pipeline from the codebase and displays the final static and dynamic outputs.

It intentionally imports the shared pipeline implementation instead of re-implementing any harmonization logic here.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 240)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "icu_data_platform").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/icu_data_platform")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from icu_data_platform.sources.asic.extract.raw_tables import (  # noqa: E402
    DEFAULT_ASIC_RAW_DATA_DIR,
    DEFAULT_TRANSLATION_PATH,
)
from icu_data_platform.sources.asic.pipeline import (  # noqa: E402
    DEFAULT_ASIC_HARMONIZED_OUTPUT_DIR,
    build_asic_harmonized_dataset,
    write_asic_harmonized_dataset,
)

PROJECT_ROOT


PosixPath('/Users/joanameyer/repository/icu-data-platform')

In [2]:
PIPELINE_PARAMETERS = {
    "raw_dir": DEFAULT_ASIC_RAW_DATA_DIR,
    "translation_path": DEFAULT_TRANSLATION_PATH,
    "min_non_null": 20,
    "min_hospitals": 4,
    "fence_factor": 1.5,
}

if not PIPELINE_PARAMETERS["raw_dir"].exists():
    raise FileNotFoundError(f"ASIC raw data directory not found: {PIPELINE_PARAMETERS['raw_dir']}")

display(Markdown("## Pipeline Parameters"))
display(pd.Series({key: str(value) for key, value in PIPELINE_PARAMETERS.items()}, name="value").to_frame())

dataset = build_asic_harmonized_dataset(**PIPELINE_PARAMETERS)
output_paths = write_asic_harmonized_dataset(dataset, PROJECT_ROOT / DEFAULT_ASIC_HARMONIZED_OUTPUT_DIR)
final_static = dataset.static.combined.copy()
final_dynamic = dataset.dynamic.combined.copy()

display(Markdown("## Saved Harmonized Outputs"))
display(pd.Series({key: str(path) for key, path in output_paths.items()}, name="path").to_frame())

display(Markdown("## Final Dataset Sizes"))
display(
    pd.DataFrame(
        [
            {
                "table": "static",
                "rows": final_static.shape[0],
                "columns": final_static.shape[1],
                "hospitals": final_static["hospital_id"].nunique(dropna=True),
            },
            {
                "table": "dynamic",
                "rows": final_dynamic.shape[0],
                "columns": final_dynamic.shape[1],
                "hospitals": final_dynamic["hospital_id"].nunique(dropna=True),
            },
        ]
    )
)


## Pipeline Parameters

,value
raw_dir,/Users/joanameyer/data/asic/synthetic_control_sample10
translation_path,/Users/joanameyer/repository/icu-data-platform/src/icu_data_platform/sources/asic/column_translation.json
min_non_null,20
min_hospitals,4
fence_factor,1.5


## Saved Harmonized Outputs

,path
static_harmonized,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/harmonized.csv
static_source_map,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/source_map.csv
static_schema_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/schema_summary.csv
static_categorical_value_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/categorical_value_summary.csv
dynamic_harmonized,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/harmonized.csv
dynamic_source_map,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/source_map.csv
dynamic_schema_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/schema_summary.csv
dynamic_non_numeric_issues,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/non_numeric_issues.csv
dynamic_semantic_decisions,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/semantic_decisions.csv
dynamic_distribution_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/distribution_summary.csv


## Final Dataset Sizes

,table,rows,columns,hospitals
0,static,80,16,8
1,dynamic,105165,107,8


## Final Static Data

`final_static` contains the harmonized static table returned by the current pipeline.


In [3]:
display(Markdown("### Preview"))
display(final_static.head(10))

display(Markdown("### Rows Per Hospital"))
display(
    final_static.groupby("hospital_id", dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)

display(Markdown("### Source Map Sample"))
display(dataset.static.source_map.head(20))

display(Markdown("### Schema Summary"))
display(dataset.static.schema_summary)


### Preview

,hospital_id,stay_id_global,stay_id_local,age_group,sex,height_group,weight_group,bmi_group,hosp_mortality,icu_mortality,hosp_los,icu_los,icu_readmit,dialysis_free_days,vent_free_days,icd10_codes
0,asic_UK00,asic_UK00_9990,9990,80-130,M,<180,76-250,Overweight,1,<NA>,<NA>,26,<NA>,<NA>,<NA>,"B96,C04,D63,E06,E87,F05,J18,J91,J95,J96,J98,K08,N39,R13,R60,R63,U69,U99,Z11,Z43"
1,asic_UK00,asic_UK00_9991,9991,70-79,F,<180,65-75,Overweight,0,<NA>,<NA>,4,<NA>,<NA>,<NA>,"C49,C78,D38,D90,J98,U99,Z11,Z85,Z92"
2,asic_UK00,asic_UK00_9992,9992,<70,M,<180,76-250,Obesity Class 3,0,<NA>,<NA>,63,<NA>,<NA>,<NA>,"D62,D69,E11,E78,E87,F10,F17,I11,I21,I25,I48,I50,J91,J95,J98,K59,N17,U69,U99,Z11,Z92"
3,asic_UK00,asic_UK00_9993,9993,80-130,M,<180,76-250,Obesity Class 1,0,<NA>,<NA>,26,<NA>,<NA>,<NA>,"A41,B96,D68,E11,E53,E79,E87,F05,F10,F13,F33,G40,G93,H10,H53,H57,I10,J13,J15,J95,J96,J98,K31,R65,S00,S01,S02,S06,U50,U52,X59,Z43"
4,asic_UK00,asic_UK00_9994,9994,70-79,M,<180,76-250,Overweight,1,<NA>,<NA>,11,<NA>,<NA>,<NA>,"C71,D43,D65,D68,E11,E66,G40,G81,G93,I10,I48,I61,I63,J95,N39,R13,R29,R47,U50,U99,Z11,Z90"
5,asic_UK00,asic_UK00_9995,9995,<70,M,<180,76-250,Obesity Class 3,1,<NA>,<NA>,9,<NA>,<NA>,<NA>,"A41,B95,D62,E03,E11,E66,E78,E87,I25,I27,I44,I48,I50,J15,J91,N17,N18,R57,R65,U69,U81,Z29,Z43,Z92,Z99"
6,asic_UK00,asic_UK00_9996,9996,<70,M,<180,65-75,Obesity Class 2,0,<NA>,<NA>,10,<NA>,<NA>,<NA>,"E11,E66,E78,E87,I10,I25,I31,I35,I48,I50,I65,J15,J18,J91,L89,N17,R13,R65,U69,U99,Z11,Z43,Z99"
7,asic_UK00,asic_UK00_9997,9997,<70,M,<180,76-250,Obesity Class 2,0,<NA>,<NA>,26,<NA>,<NA>,<NA>,"A41,B37,B96,D62,D65,D68,D69,E87,F05,I10,I71,J15,J90,J95,J96,J98,N17,N39,N40,R65,U69,U81,Z29,Z43,Z86"
8,asic_UK00,asic_UK00_9998,9998,<70,M,180-185,65-75,Overweight,1,<NA>,<NA>,22,<NA>,<NA>,<NA>,"A09,A41,D62,D90,E78,E87,F05,I10,I25,J18,J44,J91,J95,J96,J98,L89,N17,N18,R40,R65,S71,S72,U69,U99,Z03,Z11,Z74,Z86,Z92,Z95,Z99"
9,asic_UK00,asic_UK00_9999,9999,<70,F,180-185,65-75,Obesity Class 2,0,<NA>,<NA>,5,<NA>,<NA>,<NA>,"I10,I25,I26,J17,J96,R04,Z95"


### Rows Per Hospital

,hospital_id,rows
0,asic_UK00,10
1,asic_UK01,10
2,asic_UK02,10
3,asic_UK03,10
4,asic_UK04,10
5,asic_UK06,10
6,asic_UK07,10
7,asic_UK08,10


### Source Map Sample

,hospital,canonical_name,raw_source_columns_used
0,asic_UK00,hospital_id,[derived from filename]
1,asic_UK00,stay_id_global,[derived from hospital_id and stay_id_local]
2,asic_UK00,stay_id_local,"[Pseudo-ID, PseudoID]"
3,asic_UK00,age_group,[clusterAlter]
4,asic_UK00,sex,[clusterGeschlecht]
5,asic_UK00,height_group,[clusterKoerpergroesse]
6,asic_UK00,weight_group,[clusterKoerpergewicht]
7,asic_UK00,bmi_group,[BMI]
8,asic_UK00,hosp_mortality,"[Entlassgrund_(verlegt_intern,_verlegt_extern,_verstorben)]"
9,asic_UK00,icu_mortality,[Sterblichkeit]


### Schema Summary

,hospital,rows,final_columns,empty_columns
0,asic_UK00,10,16,"[icu_mortality, hosp_los, icu_readmit, dialysis_free_days, vent_free_days]"
1,asic_UK01,10,16,"[bmi_group, hosp_mortality, icu_mortality, hosp_los, icu_los, icu_readmit, dialysis_free_days, vent_free_days]"
2,asic_UK02,10,16,[]
3,asic_UK03,10,16,"[icu_mortality, hosp_los, icu_readmit, dialysis_free_days, vent_free_days]"
4,asic_UK04,10,16,"[dialysis_free_days, vent_free_days]"
5,asic_UK06,10,16,[]
6,asic_UK07,10,16,[]
7,asic_UK08,10,16,"[hosp_los, dialysis_free_days, vent_free_days]"


## Final Dynamic Data

`final_dynamic` contains the harmonized dynamic table returned by the current pipeline.


In [4]:
display(Markdown("### Preview"))
display(final_dynamic.head(10))

display(Markdown("### Key Time-Series Columns"))
dynamic_preview_columns = [
    column
    for column in ["hospital_id", "stay_id", "time", "minutes_since_admit", "heart_rate", "map", "sbp", "dbp", "spo2", "resp_rate"]
    if column in final_dynamic.columns
]
display(final_dynamic[dynamic_preview_columns].head(20))

display(Markdown("### Rows Per Hospital"))
display(
    final_dynamic.groupby("hospital_id", dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)

display(Markdown("### Schema Summary"))
display(dataset.dynamic.schema_summary)


### Preview

,hospital_id,stay_id_global,stay_id_local,time,minutes_since_admit,heart_rate,sbp,map,dbp,resp_rate,spont_resp_rate,core_temp,spo2,sao2,scvo2,cvp,fio2,feo2,peep,delta_p,insp_pressure,compliance,ie_ratio,vt,vt_per_kg_ibw,etco2,pf_ratio,pao2,paco2,ph_art,bicarbonate_art,base_excess_art,lactate_art,pap_sys,pap_mean,pap_dias,pcwp,cardiac_output_bolus,cardiac_output_cont,cardiac_index_bolus,cardiac_index_cont,stroke_volume_bolus,stroke_volume_cont,stroke_index_bolus,stroke_index_cont,svri,pvri,gedvi,evlwi,hemoglobin,hematocrit,wbc,platelets,lymph_abs,lymph_pct,inr,ptt,d_dimer,albumin,crp,pct,il6,bilirubin_total,urea,creatinine,bnp,ntprobnp,ast,alt,ldh,amylase,lipase,troponin,ck,ck_mb,fluid_balance_24h,position_therapy,ecmo,ecmo_o2,extracorp_blood_flow,extracorp_o2_flow,inhaled_no,inhaled_iloprost,dobutamine_iv_cont,epinephrine_iv_cont,norepinephrine_iv_cont,vasopressin_iv_cont,milrinone_iv_cont,levosimendan_iv_cont,terlipressin_iv_bolus,propofol_iv_cont,midazolam_iv_cont,clonidine_iv_cont,dexmedetomidine_iv_cont,ketanest_iv_cont,isoflurane_inh,sevoflurane_inh,sufentanil_iv_cont,fentanyl_iv_cont,morphine_iv_cont,rocuronium_iv_bolus,furosemide_iv_cont,hydrocortisone_iv_bolus,prednisolone_iv_bolus,dexamethasone_iv_bolus,fludrocortisone_po_bolus,sofa
0,asic_UK00,asic_UK00_9990,9990,0 days 00:00:00,0.0,102.0,142.0,71.0,64.0,NaN,NaN,36.5,94.0,NaN,NaN,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
1,asic_UK00,asic_UK00_9990,9990,0 days 00:15:00,15.0,98.0,117.0,80.0,99.0,NaN,NaN,NaN,95.0,NaN,NaN,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2,asic_UK00,asic_UK00_9990,9990,0 days 00:30:00,30.0,95.0,123.0,65.0,45.0,NaN,NaN,NaN,95.0,NaN,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,29.0,8.5,428.0,NaN,NaN,1.1,NaN,NaN,240.74624,NaN,NaN,NaN,4.960073,2.338,38.012731,NaN,NaN,21.0,40.0,279.0,NaN,8.0,NaN,104.0,77.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
3,asic_UK00,asic_UK00_9990,9990,0 days 00:45:00,45.0,90.0,139.0,64.0,86.0,NaN,NaN,NaN,96.0,NaN,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
4,asic_UK00,asic_UK00_9990,9990,0 days 01:00:00,60.0,60.0,137.0,66.5,77.0,NaN,NaN,NaN,92.0,NaN,NaN,6.0,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,238.888889,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
5,asic_UK00,asic_UK00_9990,9990,0 days 01:15:00,75.0,86.0,110.0,63.0,53.0,NaN,NaN,NaN,100.0,NaN,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>

### Key Time-Series Columns

,hospital_id,time,minutes_since_admit,heart_rate,map,sbp,dbp,spo2,resp_rate
0,asic_UK00,0 days 00:00:00,0.0,102.0,71.0,142.0,64.0,94.0,NaN
1,asic_UK00,0 days 00:15:00,15.0,98.0,80.0,117.0,99.0,95.0,NaN
2,asic_UK00,0 days 00:30:00,30.0,95.0,65.0,123.0,45.0,95.0,NaN
3,asic_UK00,0 days 00:45:00,45.0,90.0,64.0,139.0,86.0,96.0,NaN
4,asic_UK00,0 days 01:00:00,60.0,60.0,66.5,137.0,77.0,92.0,NaN
5,asic_UK00,0 days 01:15:00,75.0,86.0,63.0,110.0,53.0,100.0,NaN
6,asic_UK00,0 days 01:30:00,90.0,75.0,79.0,115.0,52.0,99.0,NaN
7,asic_UK00,0 days 01:45:00,105.0,98.0,79.0,113.0,48.5,96.0,NaN
8,asic_UK00,0 days 02:00:00,120.0,82.0,63.0,101.0,51.0,95.0,NaN
9,asic_UK00,0 days 02:15:00,135.0,72.0,68.0,102.0,62.0,94.0,NaN


### Rows Per Hospital

,hospital_id,rows
0,asic_UK00,11565
1,asic_UK01,6083
2,asic_UK02,12116
3,asic_UK03,19201
4,asic_UK04,9025
5,asic_UK06,11677
6,asic_UK07,19170
7,asic_UK08,16328


### Schema Summary

,hospital,rows,final_columns,empty_columns
0,asic_UK00,11565,107,"[feo2, cardiac_output_bolus, cardiac_index_bolus, stroke_volume_bolus, stroke_index_bolus, pvri, gedvi, evlwi, ntprobnp, ecmo_o2, extracorp_blood_flow, extracorp_o2_flow, inhaled_iloprost, dobutam..."
1,asic_UK01,6083,107,"[heart_rate, sbp, map, dbp, resp_rate, spont_resp_rate, core_temp, spo2, sao2, scvo2, cvp, fio2, feo2, compliance, ie_ratio, vt, etco2, pao2, paco2, ph_art, bicarbonate_art, base_excess_art, lacta..."
2,asic_UK02,12116,107,"[spont_resp_rate, core_temp, sao2, cvp, feo2, delta_p, vt, ph_art, pap_sys, pap_mean, pap_dias, pcwp, cardiac_output_bolus, cardiac_output_cont, cardiac_index_bolus, cardiac_index_cont, stroke_vol..."
3,asic_UK03,19201,107,"[sbp, map, dbp, scvo2, feo2, ie_ratio, pap_sys, pap_mean, pap_dias, cardiac_output_cont, cardiac_index_bolus, cardiac_index_cont, stroke_volume_bolus, stroke_volume_cont, stroke_index_bolus, strok..."
4,asic_UK04,9025,107,"[scvo2, feo2, vt, pap_sys, pap_mean, pap_dias, pcwp, cardiac_output_cont, stroke_volume_bolus, stroke_index_bolus, pvri, d_dimer, bnp, ntprobnp, fluid_balance_24h, position_therapy, ecmo, ecmo_o2,..."
5,asic_UK06,11677,107,"[heart_rate, sbp, map, dbp, resp_rate, spont_resp_rate, core_temp, spo2, sao2, scvo2, cvp, fio2, feo2, delta_p, compliance, ie_ratio, vt, vt_per_kg_ibw, etco2, pao2, paco2, ph_art, bicarbonate_art..."
6,asic_UK07,19170,107,"[feo2, compliance, cardiac_output_bolus, cardiac_output_cont, stroke_volume_bolus, stroke_volume_cont, stroke_index_bolus, stroke_index_cont, pvri, bnp, ck_mb, extracorp_blood_flow, inhaled_no, in..."
7,asic_UK08,16328,107,"[scvo2, cvp, feo2, vt, etco2, pcwp, cardiac_output_bolus, cardiac_output_cont, cardiac_index_bolus, cardiac_index_cont, stroke_volume_bolus, stroke_volume_cont, stroke_index_bolus, stroke_index_co..."


## Pipeline QC Outputs

These are the additional pipeline artifacts generated alongside the final tables.


In [5]:
display(Markdown("### Static Categorical Value Summary"))
display(dataset.static.categorical_value_summary.head(30))

display(Markdown("### Dynamic Non-Numeric Issues"))
display(dataset.dynamic.non_numeric_issues)

display(Markdown("### Dynamic Distribution Summary"))
display(dataset.dynamic.distribution_summary.head(30))

display(Markdown("### Dynamic Distribution Issues"))
display(dataset.dynamic.distribution_issues)


### Static Categorical Value Summary

,hospital_id,column,value,count
0,asic_UK00,bmi_group,Obesity Class 1,1
1,asic_UK00,bmi_group,Obesity Class 2,3
2,asic_UK00,bmi_group,Obesity Class 3,2
3,asic_UK00,bmi_group,Overweight,4
4,asic_UK01,bmi_group,<NA>,10
5,asic_UK02,bmi_group,Normal Weight,2
6,asic_UK02,bmi_group,Obesity Class 1,2
7,asic_UK02,bmi_group,Overweight,6
8,asic_UK03,bmi_group,Normal Weight,3
9,asic_UK03,bmi_group,Obesity Class 1,2


### Dynamic Non-Numeric Issues

,hospital,raw_column,canonical_name,non_null_count,raw_non_numeric_count,resolved_by_custom_parser_count,unresolved_after_custom_parser_count,non_numeric_examples
0,asic_UK02,I:E,ie_ratio,1194,1194,1194,0,"[1/2.1, 1/1.4, 1/1.5, 1/1.2, 1/1.3, 1/2, 1/1.1, 1/1, 1.2/1, 4/1]"
1,asic_UK04,Bilirubin_ges.,bilirubin_total,42,4,4,0,[<0.15]
2,asic_UK04,GPT,alt,52,1,1,0,[<5]


### Dynamic Distribution Summary

,hospital,canonical_name,n,min,q1,median,q3,max,iqr,range_width
0,asic_UK00,albumin,57,240.74624,300.932800,346.072720,421.305920,571.772320,120.373120,331.026080
1,asic_UK02,albumin,134,0.00000,338.550000,374.660000,423.935000,570.270000,85.385000,570.270000
2,asic_UK07,albumin,147,4.11000,317.400000,343.480000,381.890000,463.780000,64.490000,459.670000
3,asic_UK08,albumin,89,0.00000,286.128359,323.247785,365.007036,470.178605,78.878677,470.178605
4,asic_UK00,alt,119,9.00000,20.500000,37.000000,71.000000,401.000000,50.500000,392.000000
5,asic_UK03,alt,125,6.59900,26.995000,35.994000,71.988000,3019.897000,44.993000,3013.298000
6,asic_UK04,alt,52,0.00000,14.000000,23.000000,47.000000,323.000000,33.000000,323.000000
7,asic_UK07,alt,211,0.00000,17.500000,39.000000,62.000000,152.000000,44.500000,152.000000
8,asic_UK08,alt,182,0.00000,22.000000,33.000000,125.000000,5272.000000,103.000000,5272.000000
9,asic_UK00,ast,133,10.00000,31.000000,42.000000,78.000000,697.000000,47.000000,687.000000


### Dynamic Distribution Issues

,canonical_name,hospital,n,flagged_metrics,min,median,iqr,max,range_width
0,albumin,asic_UK00,57,[min],240.746240,346.072720,120.373120,571.772320,331.02608
1,alt,asic_UK04,52,"[median, iqr]",0.000000,23.000000,33.000000,323.000000,323.00000
2,alt,asic_UK08,182,[iqr],0.000000,33.000000,103.000000,5272.000000,5272.00000
3,base_excess_art,asic_UK00,947,[max],-9.500000,3.900000,6.700000,25.000000,34.50000
4,base_excess_art,asic_UK04,608,[max],-9.100000,2.400000,4.100000,7.200000,16.30000
...,...,...,...,...,...,...,...,...,...
58,troponin,asic_UK07,45,[median],0.010000,0.220000,0.120000,3.700000,3.69000
59,urea,asic_UK08,181,[median],0.000000,12.802080,7.795680,37.905600,37.90560
60,vt_per_kg_ibw,asic_UK00,473,"[min, iqr, max, range_width]",4.224852,4.757396,0.798817,6.248122,2.02327
61,vt_per_kg_ibw,asic_UK07,13948,"[max, range_width]",0.000000,7.070000,2.990000,38.710000,38.71000


## Working Objects

The main notebook variables are:

- `dataset`: full pipeline result object
- `final_static`: final harmonized static table
- `final_dynamic`: final harmonized dynamic table
